In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Use a non-interactive backend for clusters or remote execution
import matplotlib.pyplot as plt
import seaborn as sns
from skimage import io
from scipy.ndimage import gaussian_filter
from skimage.filters import threshold_otsu
import glob
import os
import re
from scipy.stats import pearsonr
import pandas as pd
from tqdm import tqdm
import warnings
import string
from scipy.stats import ConstantInputWarning
import logging
import hashlib
from joblib import Parallel, delayed
import json
import h5py
import time

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def ensure_dir(directory):
    try:
        os.makedirs(directory, exist_ok=True)
    except FileExistsError:
        pass

def sanitize_filename(filename):
    valid_chars = "-_. %s%s" % (string.ascii_letters, string.digits)
    sanitized = ''.join(c for c in filename if c in valid_chars)
    return sanitized

def hash_mask_name(mask_name, length=8):
    hash_object = hashlib.sha256(mask_name.encode('utf-8'))
    hex_dig = hash_object.hexdigest()
    return hex_dig[:length]

def cross_correlation_coefficient(image1, image2):
    img1_flat = image1.flatten()
    img2_flat = image2.flatten()
    mask = ~np.isnan(img1_flat) & ~np.isnan(img2_flat)
    img1_flat = img1_flat[mask]
    img2_flat = img2_flat[mask]
    if len(img1_flat) == 0 or len(img2_flat) == 0:
        return np.nan
    if np.all(img1_flat == img1_flat[0]) or np.all(img2_flat == img2_flat[0]):
        return np.nan
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=ConstantInputWarning)
        coefficient, _ = pearsonr(img1_flat, img2_flat)
    return coefficient

def blur_and_threshold(image_path, results_dir, sigma=2):
    base_name = os.path.splitext(os.path.basename(image_path))[0]
    
    # Define directories for processed images
    blurred_dir = os.path.join(results_dir, 'processed_images', f'blurred_sigma{sigma}')
    thresholded_dir = os.path.join(results_dir, 'processed_images', f'thresholded_sigma{sigma}')
    symmetry_dir = os.path.join(results_dir, 'processed_images', f'symmetry_sigma{sigma}')
    ensure_dir(blurred_dir)
    ensure_dir(thresholded_dir)
    ensure_dir(symmetry_dir)

    blurred_filename = f'{base_name}_blurred_sigma{sigma}.tif'
    thresholded_filename = f'{base_name}_thresholded_sigma{sigma}.tif'
    blurred_path = os.path.join(blurred_dir, blurred_filename)
    thresholded_path = os.path.join(thresholded_dir, thresholded_filename)

    left_filename = f'{base_name}_left_sigma{sigma}.tif'
    mirrored_right_filename = f'{base_name}_mirrored_right_sigma{sigma}.tif'
    left_path = os.path.join(symmetry_dir, left_filename)
    mirrored_right_path = os.path.join(symmetry_dir, mirrored_right_filename)

    stack_thresholded = None
    left_half = None
    mirrored_right_half = None

    try:
        if os.path.exists(thresholded_path):
            print(f"[Sigma {sigma}] Loading existing thresholded image: {thresholded_path}")
            stack_thresholded = io.imread(thresholded_path).astype(np.float64)
        else:
            print(f"[Sigma {sigma}] Processing new image: {image_path}")
            stack = io.imread(image_path).astype(np.float64)
            if stack is None:
                print(f"Warning: Failed to load image {image_path}")
                return None, None, None
            stack_bg = stack[0:10, 0:10, 0:10].mean()
            stack -= stack_bg
            stack = np.clip(stack, 0, None)
            stack_blurred = gaussian_filter(stack, sigma=sigma)
            thresh = threshold_otsu(stack_blurred)
            stack_thresholded = np.where(stack_blurred > thresh, stack_blurred, 0)
            
            if save_images and not os.path.exists(blurred_path):
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    io.imsave(blurred_path, np.clip(stack_blurred, 0, 255).astype(np.uint8))

            if save_images:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    io.imsave(thresholded_path, np.clip(stack_thresholded, 0, 255).astype(np.uint8))
                print(f"[Sigma {sigma}] Saved thresholded image: {thresholded_path}")


        if os.path.exists(left_path) and os.path.exists(mirrored_right_path):
            print(f"[Sigma {sigma}] Loading existing symmetry images for: {image_path}")
            left_half = io.imread(left_path).astype(np.float64)
            mirrored_right_half = io.imread(mirrored_right_path).astype(np.float64)
        else:
            print(f"[Sigma {sigma}] Creating symmetry images for: {image_path}")
            left_half, mirrored_right_half = split_image(stack_thresholded)
            
            if save_images:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    io.imsave(left_path, np.clip(left_half, 0, 255).astype(np.uint8))
                    io.imsave(mirrored_right_path, np.clip(mirrored_right_half, 0, 255).astype(np.uint8))
                print(f"[Sigma {sigma}] Saved symmetry images for: {image_path}")
    except Exception as e:
        print(f"Error processing image {image_path}: {e}")
        return None, None, None

    return stack_thresholded, left_half, mirrored_right_half

def calculate_cross_correlation(processed_images, left_halves, mirrored_right_halves):
    n = len(processed_images)
    correlation_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            if i == j:
                coefficient = cross_correlation_coefficient(left_halves[i], mirrored_right_halves[i])
                correlation_matrix[i, j] = correlation_matrix[j, i] = coefficient
            else:
                coefficient = cross_correlation_coefficient(processed_images[i], processed_images[j])
                correlation_matrix[i, j] = correlation_matrix[j, i] = coefficient
    return correlation_matrix

def plot_heatmap(matrix, gene_name, save_dir, sigma, suffix=''):
    plt.figure(figsize=(10, 8))
    sns.heatmap(matrix, annot=True, fmt=".2f", cmap="viridis", vmin=-1, vmax=1)
    plt.title(f'Cross-Correlation Heatmap for {gene_name} sigma={sigma} {suffix}')
    ensure_dir(save_dir)
    filename = os.path.join(save_dir, f'{gene_name}_heatmap_sigma{sigma}{suffix}.png')
    if os.path.exists(filename):
        print(f"[Sigma {sigma}] Heatmap already exists: {filename}")
        plt.close()
        return
    if save_images:
        plt.savefig(filename)
        print(f"[Sigma {sigma}] Saved heatmap: {filename}")
    plt.close()

def save_correlation_matrix_h5(matrix, gene_name, save_dir, sigma, suffix=''):
    ensure_dir(save_dir)
    h5_file = os.path.join(save_dir, 'cross_correlation_results.h5')
    group_name = f'{gene_name}/sigma_{sigma}{suffix}'
    with h5py.File(h5_file, 'a') as f:
        if group_name in f:
            print(f"[Sigma {sigma}] Correlation matrix already exists for {group_name}")
            return
        f.create_dataset(group_name, data=matrix)
        print(f"[Sigma {sigma}] Saved correlation matrix for {group_name}")

def split_image(image, split_index=282):
    # Split the image into left and right halves
    left_half = image[:, :, :split_index]
    right_half = image[:, :, split_index:]
    
    # Ensure both halves have the same width by padding
    max_width = max(left_half.shape[2], right_half.shape[2])
    if left_half.shape[2] != max_width:
        pad_width = max_width - left_half.shape[2]
        left_half = np.pad(left_half, ((0, 0), (0, 0), (pad_width, 0)), 'constant')
    if right_half.shape[2] != max_width:
        pad_width = max_width - right_half.shape[2]
        right_half = np.pad(right_half, ((0, 0), (0, 0), (0, pad_width)), 'constant')
    
    # Mirror the right half
    mirrored_right_half = np.flip(right_half, axis=2)
    
    return left_half, mirrored_right_half

save_images = False  # Set to True if you want to save images

# Define your image directory paths
base_dir = r'./mapzbrain_analysis'
raw_image_dir = os.path.join(base_dir, 'images')
mask_dir = os.path.join(base_dir, 'chosen_masks')
mask_paths = glob.glob(os.path.join(mask_dir, '*.tif'))[:]  # Limiting to 3 masks for test
mask_names = [os.path.splitext(os.path.basename(p))[0] for p in mask_paths]
file_paths = glob.glob(os.path.join(raw_image_dir, '*.tif'))
results_dir = os.path.join(base_dir, 'cross_corr_results_parallel')
ensure_dir(results_dir)
checkpoint_file = os.path.join(results_dir, 'checkpoint.json')
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'r') as f:
        processed_data = json.load(f)
else:
    processed_data = {}
# Define result subdirectories
processed_images_dir = os.path.join(results_dir, 'processed_images')
heatmap_dir = os.path.join(results_dir, 'heatmaps')
correlation_dir = os.path.join(results_dir, 'correlation_matrices')
ensure_dir(processed_images_dir)
ensure_dir(heatmap_dir)
ensure_dir(correlation_dir)

gene_files = {}
for file_path in file_paths:
    match = re.match(r'T_(\w+)_\d+.tif', os.path.basename(file_path))
    if match:
        gene_name = match.group(1)
        gene_files.setdefault(gene_name, []).append(file_path)

sigma_values = [0,2,4,6,8,10]

# Save mask hash mapping to JSON for later reinterpretation
mask_hash_mapping = {sanitize_filename(mask_name): hash_mask_name(mask_name) for mask_name in mask_names}
hash_mapping_file = os.path.join(results_dir, 'mask_hash_mapping.json')
with open(hash_mapping_file, 'w') as f:
    json.dump(mask_hash_mapping, f)

# Initialize lists to store overall results
overall_masked_results_list = []
overall_masked_symmetry_results_list = []

# Iterate over genes
for gene, paths in tqdm(gene_files.items(), desc='Genes'):
    if gene not in processed_data:
        processed_data[gene] = {}  # Initialize entry for the gene if not present
    for sigma in sigma_values:
        if str(sigma) in processed_data[gene]:
            print(f"Skipping sigma {sigma} for gene {gene}, already processed.")
            continue

        print(f"Processing sigma {sigma} for gene {gene}")

    start_time = time.time()
    print(f'Starting processing for gene: {gene}')
    processed_data[gene] = {}
    for sigma in sigma_values:
        if str(sigma) in processed_data[gene]:
            print(f"Skipping sigma {sigma} for gene {gene}, already processed.")
            continue

        print(f"Processing sigma {sigma} for gene {gene}")

        def process_image_parallel(path):
            return blur_and_threshold(path, results_dir, sigma=sigma)

        results = Parallel(n_jobs=-1, backend="threading")(delayed(process_image_parallel)(path) for path in paths)

        # Filter out any failed results
        results = [result for result in results if result[0] is not None]
        if not results:
            print(f"Warning: No valid images processed for gene: {gene}, sigma: {sigma}")
            continue

        processed_images, left_halves, mirrored_right_halves = zip(*results)
        # Check if correlation matrix already exists
        h5_file = os.path.join(correlation_dir, 'cross_correlation_results.h5')
        group_name = f'{gene}/sigma_{sigma}'
        with h5py.File(h5_file, 'a') as f:
            if group_name in f:
                print(f"[Sigma {sigma}] Correlation matrix already exists for {group_name}")
                correlation_matrix = f[group_name][()]
            else:
                print(f"[Sigma {sigma}] Calculating correlation matrix for {gene}, sigma {sigma}")
                correlation_matrix = calculate_cross_correlation(processed_images, left_halves, mirrored_right_halves)
                f.create_dataset(group_name, data=correlation_matrix)
                print(f"[Sigma {sigma}] Saved correlation matrix for {group_name}")

        # Plot heatmap
        heatmap_filename = os.path.join(heatmap_dir, f'{gene}_heatmap_sigma{sigma}.png')
        if os.path.exists(heatmap_filename):
            print(f"[Sigma {sigma}] Heatmap already exists: {heatmap_filename}")
        else:
            plot_heatmap(correlation_matrix, gene, heatmap_dir, sigma)
            print(f"[Sigma {sigma}] Saved heatmap: {heatmap_filename}")

        processed_data[gene][str(sigma)] = True

        # Now apply masks to images, calculate masked cross-correlations and symmetry
        for mask_path, mask_name in tqdm(zip(mask_paths, mask_names), total=len(mask_paths), desc=f'Masks (Sigma {sigma})', leave=False):
            mask = io.imread(mask_path).astype(bool)
            sanitized_mask_name = sanitize_filename(mask_name)
            # Generate a hash for the mask name
            hashed_mask_name = hash_mask_name(sanitized_mask_name)
            # Combine sanitized name and hash for readability
            short_mask_name = f"{sanitized_mask_name[:10]}_{hashed_mask_name}"
            # Apply mask to processed images
            masked_processed_images = []
            masked_left_halves = []
            masked_mirrored_right_halves = []
            
            # Directory to save masked images
            masked_images_dir = os.path.join(results_dir, 'masked', f'mask_{short_mask_name}', f'sigma_{sigma}')
            ensure_dir(masked_images_dir)
            
            for idx, img in enumerate(processed_images):
                # Prepare filenames and paths
                img_filename = os.path.basename(paths[idx])
                base_filename = os.path.splitext(img_filename)[0]
                masked_img_filename = f'{base_filename}_masked_{short_mask_name}_sigma{sigma}.tif'
                masked_img_path = os.path.join(masked_images_dir, masked_img_filename)
                
                # Check if masked image already exists
                if os.path.exists(masked_img_path):
                    print(f"[Sigma {sigma}] [Mask {mask_name}] Loading existing masked image: {masked_img_path}")
                    masked_img = io.imread(masked_img_path).astype(np.float64)
                    masked_img[masked_img == 0] = np.nan  # Replace zeros with NaN
                    masked_processed_images.append(masked_img)
                else:
                    # Resize mask to match image shape
                    if img.shape != mask.shape:
                        from skimage.transform import resize
                        mask_resized = resize(mask, img.shape, preserve_range=True, order=0).astype(bool)
                    else:
                        mask_resized = mask

                    # Apply mask to full image
                    masked_img = img.copy()
                    masked_img[~mask_resized] = np.nan  # Set outside mask to NaN
                    masked_processed_images.append(masked_img)

                    # Save masked full image
                    if save_images:
                        with warnings.catch_warnings():
                            warnings.simplefilter("ignore")
                            io.imsave(masked_img_path, np.nan_to_num(masked_img, nan=0).astype(np.uint8))
                        print(f"[Sigma {sigma}] [Mask {mask_name}] Saved masked image: {masked_img_path}")


                # Now process the masked left and right halves
                # Left half
                masked_left_filename = f'{base_filename}_left_masked_{short_mask_name}_sigma{sigma}.tif'
                masked_left_path = os.path.join(masked_images_dir, masked_left_filename)
                if os.path.exists(masked_left_path):
                    print(f"[Sigma {sigma}] [Mask {mask_name}] Loading existing masked left half: {masked_left_path}")
                    masked_left = io.imread(masked_left_path).astype(np.float64)
                    masked_left[masked_left == 0] = np.nan  # Replace zeros with NaN
                    masked_left_halves.append(masked_left)
                else:
                    # Apply mask to left half
                    if 'mask_resized' not in locals():
                        # Ensure mask_resized is available
                        if img.shape != mask.shape:
                            from skimage.transform import resize
                            mask_resized = resize(mask, img.shape, preserve_range=True, order=0).astype(bool)
                        else:
                            mask_resized = mask

                    left_mask, mirrored_right_mask = split_image(mask_resized)

                    masked_left = left_halves[idx].copy()
                    masked_left[~left_mask] = np.nan  # Set outside mask to NaN
                    masked_left_halves.append(masked_left)
                    if save_images:
                        with warnings.catch_warnings():
                            warnings.simplefilter("ignore")
                            io.imsave(masked_left_path, np.nan_to_num(masked_left, nan=0).astype(np.uint8))
                        print(f"[Sigma {sigma}] [Mask {mask_name}] Saved masked left half: {masked_left_path}")

                # Mirrored right half
                masked_right_filename = f'{base_filename}_right_masked_{short_mask_name}_sigma{sigma}.tif'
                masked_right_path = os.path.join(masked_images_dir, masked_right_filename)
                if os.path.exists(masked_right_path):
                    print(f"[Sigma {sigma}] [Mask {mask_name}] Loading existing masked right half: {masked_right_path}")
                    masked_right = io.imread(masked_right_path).astype(np.float64)
                    masked_right[masked_right == 0] = np.nan  # Replace zeros with NaN
                    masked_mirrored_right_halves.append(masked_right)
                else:
                    # Apply mask to mirrored right half
                    if 'mask_resized' not in locals():
                        # Ensure mask_resized is available
                        if img.shape != mask.shape:
                            from skimage.transform import resize
                            mask_resized = resize(mask, img.shape, preserve_range=True, order=0).astype(bool)
                        else:
                            mask_resized = mask

                    left_mask, mirrored_right_mask = split_image(mask_resized)

                    masked_right = mirrored_right_halves[idx].copy()
                    masked_right[~mirrored_right_mask] = np.nan  # Set outside mask to NaN
                    masked_mirrored_right_halves.append(masked_right)
                    if save_images:
                        with warnings.catch_warnings():
                            warnings.simplefilter("ignore")
                            io.imsave(masked_right_path, np.nan_to_num(masked_right, nan=0).astype(np.uint8))
                        print(f"[Sigma {sigma}] [Mask {mask_name}] Saved masked right half: {masked_right_path}")
                        
            # Calculate cross-correlation matrix for masked images
            masked_correlation_dir = os.path.join(correlation_dir, f'masked_{short_mask_name}')
            ensure_dir(masked_correlation_dir)
            h5_file = os.path.join(masked_correlation_dir, 'cross_correlation_results.h5')
            group_name = f'{gene}/sigma_{sigma}_masked_{short_mask_name}'
            with h5py.File(h5_file, 'a') as f:
                if group_name in f:
                    print(f"[Sigma {sigma}] [Mask {mask_name}] Masked correlation matrix already exists for {group_name}")
                    masked_correlation_matrix = f[group_name][()]
                else:
                    print(f"[Sigma {sigma}] [Mask {mask_name}] Calculating masked correlation matrix for {gene}, sigma {sigma}")
                    masked_correlation_matrix = calculate_cross_correlation(masked_processed_images, masked_left_halves, masked_mirrored_right_halves)
                    f.create_dataset(group_name, data=masked_correlation_matrix)
                    print(f"[Sigma {sigma}] [Mask {mask_name}] Saved masked correlation matrix for {group_name}")

            # Plot and save the masked correlation heatmap
            masked_heatmap_dir = os.path.join(heatmap_dir, f'masked_{short_mask_name}')
            ensure_dir(masked_heatmap_dir)
            heatmap_filename = os.path.join(masked_heatmap_dir, f'{gene}_heatmap_sigma{sigma}_masked_{short_mask_name}.png')
            if os.path.exists(heatmap_filename):
                print(f"[Sigma {sigma}] [Mask {mask_name}] Heatmap already exists: {heatmap_filename}")
            else:
                plot_heatmap(masked_correlation_matrix, gene, masked_heatmap_dir, sigma, suffix=f'_masked_{short_mask_name}')
                print(f"[Sigma {sigma}] [Mask {mask_name}] Saved heatmap: {heatmap_filename}")

            # Populate masked results DataFrame
            for i in range(len(paths)):
                for j in range(i + 1, len(paths)):
                    row = {
                        'Gene': gene,
                        'Sigma': sigma,
                        'Mask': mask_name,  # Original mask name
                        'MaskHash': hashed_mask_name,  # Hashed mask name
                        'Image1': os.path.basename(paths[i]),
                        'Image2': os.path.basename(paths[j]),
                        'MaskedCrossCorrelation': masked_correlation_matrix[i, j]
                    }
                    overall_masked_results_list.append(row)

            # Populate masked symmetry results DataFrame
            for i, path in enumerate(paths):
                row = {
                    'Gene': gene,
                    'Sigma': sigma,
                    'Mask': mask_name,
                    'MaskHash': hashed_mask_name,
                    'Image': os.path.basename(path),
                    'MaskedSymmetryCrossCorrelation': masked_correlation_matrix[i, i]
                }
                overall_masked_symmetry_results_list.append(row)

        with open(checkpoint_file, 'w') as f:
            json.dump(processed_data, f)
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f'Finished processing for gene: {gene} in {elapsed_time:.2f} seconds')

# Save overall masked results to CSV
overall_masked_results_df = pd.DataFrame(overall_masked_results_list)
overall_masked_results_df.to_csv(os.path.join(results_dir, 'overall_masked_cross_correlation_results.csv'), index=False)

overall_masked_symmetry_results_df = pd.DataFrame(overall_masked_symmetry_results_list)
overall_masked_symmetry_results_df.to_csv(os.path.join(results_dir, 'overall_masked_symmetry_results.csv'), index=False)

print('Processing complete')
